In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np

# Example: create dummy predictions
# Replace `test.csv` with your actual test file if needed

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("WANDB_API_KEY")


In [5]:
!pip install -q transformers datasets accelerate wandb evaluate

In [7]:
import wandb


os.environ["WANDB_ENTITY"] = "23f3003322-indian-institute-of-technology-madras"
os.environ["WANDB_PROJECT"] = "23f3003322-t22026"
os.environ["WANDB_LOG_MODEL"] = "false"  # Set to "end" if you want checkpoint artifacts uploaded

wandb.login(key=secret_value_1)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [9]:
import torch
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForMultipleChoice, 
    TrainingArguments, 
    Trainer,
    PreTrainedTokenizerBase
)
from transformers.tokenization_utils_base import PaddingStrategy

# Verify GPU
print(f"Using GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU found!")

Using GPU: Tesla T4


In [22]:
# 2. Initialize WandB Run
run = wandb.init(
    entity="23f3003322-indian-institute-of-technology-madras",
    project="23f3003322-t22026",
    name="deberta-v3-base-run-2",
    group="model-1-deberta",
    job_type="train",
    tags=["deberta-v3", "multiple-choice", "fine-tuning", "model-1"],
    config={
        "model_architecture": "AutoModelForMultipleChoice",
        "model_name": "microsoft/deberta-v3-base",
        "max_length": 256,
        "batch_size": 4,
        "gradient_accumulation_steps": 4,  # Effective batch size = 16
        "learning_rate": 2e-5,
        "epochs": 5,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "fp16": True,
        "optimizer": "AdamW",
    }
)

config = wandb.config

In [23]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

option_to_index = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
index_to_option = {v: k for k, v in option_to_index.items()}

def preprocess_function(examples):
    # Repeat prompt 5 times for each row
    first_sentences = [[prompt] * 5 for prompt in examples["prompt"]]
    
    # Collect options A through E
    second_sentences = [
        [examples[opt][i] for opt in ["A", "B", "C", "D", "E"]]
        for i in range(len(examples["prompt"]))
    ]

    # Flatten for tokenization
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    # Tokenize pairs
    tokenized_examples = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=config.max_length,
    )

    # Reshape back to (batch_size, 5, max_length)
    tokenized_inputs = {
        k: [v[i : i + 5] for i in range(0, len(v), 5)]
        for k, v in tokenized_examples.items()
    }
    
    # Map label letter to integer index (if label exists in dataset)
    if "answer" in examples:
        tokenized_inputs["labels"] = [option_to_index[ans] for ans in examples["answer"]]
        
    return tokenized_inputs

In [24]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features] if label_name in features[0] else None
        
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        # Unflatten back to (batch_size, num_choices, max_seq_length)
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
            
        return batch

In [25]:
def mapk(targets, predictions, k=3):
    """Calculates Mean Average Precision at K for multiple choice evaluation."""
    scores = []
    for target, preds in zip(targets, predictions):
        score = 0.0
        for i, p in enumerate(preds[:k]):
            if p == target:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    # Standard Accuracy (Top 1)
    preds_top1 = np.argmax(logits, axis=-1)
    acc = np.mean(preds_top1 == labels)
    
    # Top 3 Ranked Predictions for MAP@3
    top3_preds = np.argsort(-logits, axis=1)[:, :3]
    map3_score = mapk(labels, top3_preds, k=3)
    
    return {
        "accuracy": acc,
        "map@3": map3_score
    }

In [19]:
from sklearn.model_selection import train_test_split

# 1. Load the training data
df_full = train.copy()

# 2. Perform a train/val split (85% train, 15% validation)
df_train, df_val = train_test_split(
    df_full, 
    test_size=0.15, 
    random_state=42,
    stratify=df_full["answer"] if "answer" in df_full else None # Keeps answer distribution balanced
)

print(f"Training samples: {len(df_train)} | Validation samples: {len(df_val)}")

# 3. Map preprocessing
train_dataset = Dataset.from_pandas(df_train).map(preprocess_function, batched=True)
val_dataset = Dataset.from_pandas(df_val).map(preprocess_function, batched=True)

Training samples: 1700 | Validation samples: 300


Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [26]:
# Replace 'train.csv' and 'val.csv' with your local/Kaggle dataset paths
# df_train = Dataset.from_pandas(df_train).map(preprocess_function, batched=True)
# df_val = Dataset.from_pandas(df_val).map(preprocess_function, batched=True)

# train_dataset = Dataset.from_pandas(df_train).map(preprocess_function, batched=True)
# val_dataset = Dataset.from_pandas(df_val).map(preprocess_function, batched=True)

# Load pretrained DeBERTa model
model = AutoModelForMultipleChoice.from_pretrained(config.model_name)

# Ensure all trainable parameters are in fp32 for AMP
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
# ------------------------

training_args = TrainingArguments(
    output_dir="./deberta_mcq_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=config.learning_rate,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    num_train_epochs=config.epochs,
    weight_decay=0.01,
    warmup_ratio=config.warmup_ratio,
    fp16=True,  # Crucial for T4 GPU memory efficiency
    report_to="wandb",
    load_best_model_at_end=True,
    metric_for_best_model="map@3",
    greater_is_better=True,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Start training & push run data to WandB
trainer.train()
wandb.finish()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                

Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,12.143851,2.712152,0.500000,0.646667
2,9.211235,1.597017,0.720000,0.817778
3,6.909623,1.252370,0.790000,0.859444
4,5.475653,1.042676,0.833333,0.881667
5,4.960381,0.985308,0.830000,0.886111


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

eval/accuracy,▁▆▇██
eval/loss,█▃▂▁▁
eval/map@3,▁▆▇██
eval/runtime,█▁▂▁▁
eval/samples_per_second,▁█▇██
eval/steps_per_second,▁█▇██
train/epoch,▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇████
train/grad_norm,▂▂▁▂▂▃▄▄▃▄▄▆▄█▇▄▄█▇▄▅▅▄▅▇▅▂
train/learning_rate,▃▆██▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▁▁
+1,...


In [ ]:
def generate_kaggle_submission(test_df, trainer, output_filename="submission.csv"):
    test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)
    predictions = trainer.predict(test_dataset)
    
    # Sort logits descending to get ranked predictions (A, B, C, D, E mapped to letters)
    top3_indices = np.argsort(-predictions.predictions, axis=1)[:, :3]
    
    preds_str = [
        " ".join([index_to_option[idx] for idx in row]) 
        for row in top3_indices
    ]
    
    submission_df = pd.DataFrame({
        "id": test_df["id"],
        "prediction": preds_str
    })
    
    submission_df.to_csv(output_filename, index=False)
    print(f"Successfully saved submission to {output_filename}")
    return submission_df

In [ ]:
# Missing values
print("Train missing:\n", train.isnull().sum())
print("Test missing:\n", test.isnull().sum())

# Data types
print(train.dtypes)

# Verify answer column exists only in train
print("Columns train:", train.columns.tolist())
print("Columns test:", test.columns.tolist())

In [ ]:
ax = sns.countplot(data=train, x='answer', order=['A','B','C','D','E'])
ax.bar_label(ax.containers[0])
plt.title("Answer Distribution in Training Set")
plt.show()

# Proportion
print(train['answer'].value_counts(normalize=True).sort_index())

In [ ]:
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

# Character lengths
for col in text_cols:
    train[col+'_char_len'] = train[col].astype(str).str.len()
    test[col+'_char_len'] = test[col].astype(str).str.len()

# Plot prompt lengths
fig, axes = plt.subplots(1,2, figsize=(12,4))
train['prompt_char_len'].hist(ax=axes[0], bins=30)
axes[0].set_title("Train Prompt Character Lengths")
test['prompt_char_len'].hist(ax=axes[1], bins=30)
axes[1].set_title("Test Prompt Character Lengths")
plt.show()

# Print statistics
for col in ['prompt_char_len'] + [c+'_char_len' for c in ['A','B','C','D','E']]:
    print(f"{col}: min={train[col].min()}, max={train[col].max()}, mean={train[col].mean():.0f}, 95th={train[col].quantile(0.95)}")

In [ ]:
train.groupby('answer')[['A_char_len','B_char_len','C_char_len','D_char_len','E_char_len']].mean()

In [ ]:
# Clean all text columns: strip, collapse whitespace
for col in text_cols:
    train[col] = train[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
    test[col] = test[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

# Optionally, remove leading/trailing weird characters if any (unlikely)
# For now, no lowercasing – we'll keep case for pretrained models.

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter

def clean_text(text):
    """
    Standardizes text by lowercasing and removing non-alphanumeric noise.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Remove HTML tags if any exist in the Kaggle prompts
    text = re.sub(r'<[^>]+>', ' ', text)
    # Keep only alphanumeric characters and standard punctuation
    text = re.sub(r'[^a-z0-9\s.,!?]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

class ScratchVocabulary:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        # Reserve index 0 for Padding and index 1 for Unknown words
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2

    def build_vocab(self, texts):
        """
        Calculates word frequencies and builds the integer mapping.
        """
        word_counts = Counter()
        for text in texts:
            words = text.split()
            word_counts.update(words)

        for word, count in word_counts.items():
            if count >= self.min_freq and word not in self.word2idx:
                self.word2idx[word] = self.vocab_size
                self.idx2word[self.vocab_size] = word
                self.vocab_size += 1

    def numericalize(self, text):
        """
        Converts a raw string into a list of integers based on the vocab.
        """
        return [self.word2idx.get(word, self.word2idx['<UNK>']) for word in text.split()]

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, df, vocab, max_length=128, is_test=False):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_length = max_length
        self.is_test = is_test

        # Label mapping as required by the Kaggle task
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def pad_or_truncate(self, tokenized_sequence):
        """
        Ensures all sequences are exactly 'max_length' for PyTorch batching.
        """
        if len(tokenized_sequence) > self.max_length:
            # Truncate
            return tokenized_sequence[:self.max_length]
        else:
            # Pad with 0s (<PAD> token)
            padding_length = self.max_length - len(tokenized_sequence)
            return tokenized_sequence + [self.vocab.word2idx['<PAD>']] * padding_length

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Clean the main prompt
        clean_prompt = clean_text(row['prompt'])

        option_tensors = []

        # Process all 5 options independently against the prompt
        for option_col in ['A', 'B', 'C', 'D', 'E']:
            clean_opt = clean_text(row[option_col])

            # Combine Prompt + Option (Context Pairing)
            combined_text = clean_prompt + " " + clean_opt

            # Convert to integers using our Scratch Vocabulary
            tokenized = self.vocab.numericalize(combined_text)

            # Pad or Truncate
            processed_seq = self.pad_or_truncate(tokenized)
            option_tensors.append(processed_seq)

        # Convert list of 5 sequences into a single PyTorch tensor (Shape: 5 x max_length)
        x_tensor = torch.tensor(option_tensors, dtype=torch.long)

        if self.is_test:
            # Test set does not have an 'answer' column
            return x_tensor, row['id']
        else:
            # Convert the correct string answer ('A', 'B', etc.) to an integer (0-4)
            label = self.label_map[row['answer']]
            y_tensor = torch.tensor(label, dtype=torch.long)
            return x_tensor, y_tensor

In [ ]:
# 1. Load the data (Assuming you have train.csv loaded via Pandas)
# train_df = pd.read_csv('train.csv')

# 2. Extract all text to build the vocabulary
print("Cleaning text and building vocabulary...")
all_texts = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    all_texts.extend(train[col].apply(clean_text).tolist())

# 3. Initialize and build the custom Vocabulary
vocab = ScratchVocabulary(min_freq=2)
vocab.build_vocab(all_texts)
print(f"Total Custom Vocabulary Size: {vocab.vocab_size}")

# 4. Create PyTorch Dataset & DataLoader
# We use max_length=128 to keep memory usage efficient
train_dataset = MCQDataset(train, vocab, max_length=128, is_test=False)

# DataLoader will automatically batch our data.
# batch_size=16 means it processes 16 questions (16 x 5 options) at a time.
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# 5. Sanity Check: Look at one batch
for batch_x, batch_y in train_loader:
    print(f"Input batch shape: {batch_x.shape}") # Expected: [16, 5, 128] -> [Batch, Options, Seq_Length]
    print(f"Target batch shape: {batch_y.shape}") # Expected: [16] -> The correct integer indices
    break

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ScratchLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, dropout=0.2):
        super(ScratchLSTMAttention, self).__init__()

        # 1. The Embedding Layer
        # Initializes random vectors for each word in your custom vocabulary.
        # padding_idx=0 ensures the <PAD> token doesn't affect gradients.
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                                      embedding_dim=embedding_dim,
                                      padding_idx=0)

        # 2. The Bi-LSTM Core
        # bidirectional=True helps the model understand context from both directions
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Because it's bidirectional, the output hidden dimension is doubled
        self.lstm_out_dim = hidden_dim * 2

        # 3. The Custom Attention Mechanism
        # A simple linear layer to calculate raw alignment scores (e_i)
        self.attention_fc = nn.Linear(self.lstm_out_dim, 1)

        # 4. The Classification Head
        # Takes the final focused context vector and outputs a single score
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.lstm_out_dim, 1)

    def forward(self, x):
        # Input 'x' comes from your MCQDataset.
        # Shape: [batch_size, 5 (options), max_length]
        batch_size, num_options, max_length = x.size()

        # Reshape to process all 5 prompt-option pairs simultaneously
        # Shape becomes: [batch_size * 5, max_length]
        x = x.view(batch_size * num_options, max_length)

        # 1. Embedding
        # Shape: [batch_size * 5, max_length, embedding_dim]
        embedded = self.embedding(x)

        # 2. LSTM Processing
        # lstm_out contains the hidden states (h_i) for every word in the sequence
        # Shape: [batch_size * 5, max_length, lstm_out_dim]
        lstm_out, _ = self.lstm(embedded)

        # 3. Attention Calculation
        # Calculate raw scores (e_i) -> Shape: [batch_size * 5, max_length, 1]
        attn_weights = self.attention_fc(lstm_out)

        # Apply softmax across the sequence length (dim=1) to get (\alpha_i)
        # This ensures all attention weights sum to 1
        attn_weights = F.softmax(attn_weights, dim=1)

        # Multiply hidden states by attention weights: \alpha_i * h_i
        # Sum across the sequence (dim=1) to get the Context Vector (c)
        # Shape: [batch_size * 5, lstm_out_dim]
        context_vector = torch.sum(lstm_out * attn_weights, dim=1)

        # 4. Classification
        context_vector = self.dropout(context_vector)

        # Get a single logit score for each Prompt+Option pair
        # Shape: [batch_size * 5, 1]
        logits = self.classifier(context_vector)

        # Reshape back to [batch_size, 5] so each of the 5 options gets its own score
        logits = logits.view(batch_size, num_options)

        # Output raw logits (CrossEntropyLoss will apply the final softmax internally)
        return logits

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Local Hyperparameter Configuration (Replacing WandB Config)
EPOCHS = 5
LEARNING_RATE = 2e-5
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
DROPOUT_RATE = 0.3

# 2. Instantiate the Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Assuming 'vocab' and 'train_loader' are already built from your preprocessing steps
model = ScratchLSTMAttention(
    vocab_size=vocab.vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT_RATE
).to(device)

# 3. Setup Loss and Optimizer
# CrossEntropyLoss automatically handles the softmax mapping to the 5 answer indices
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Model successfully loaded on {device}. Starting local training loop...\n")
print("-" * 50)

# 4. The Standard Training Loop
for epoch in range(EPOCHS):
    model.train()  # Set the model to training mode

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # Iterate through batches of (Prompt+Options, Labels)
    for batch_x, batch_y in train_loader:
        # Move tensors to the active device (CPU or GPU)
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        # Clear old gradients from the previous step
        optimizer.zero_grad()

        # Forward Pass: Get the logits for all 5 options
        logits = model(batch_x)

        # Compute the loss
        loss = criterion(logits, batch_y)

        # Backward Pass: Compute gradients
        loss.backward()

        # Update weights
        optimizer.step()

        # --- Local Metric Tracking ---
        total_loss += loss.item()

        # Calculate training accuracy for this batch
        predictions = torch.argmax(logits, dim=1)
        correct_predictions += (predictions == batch_y).sum().item()
        total_samples += batch_y.size(0)

    # Calculate average metrics for the epoch
    avg_loss = total_loss / len(train_loader)
    epoch_accuracy = correct_predictions / total_samples

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_loss:.4f} | Train Accuracy: {epoch_accuracy:.4f}")

print("-" * 50)
print("Training Complete!")

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
import torch
import numpy as np

# 1. Load the Model and Tokenizer
model_name = "roberta-base" # We can upgrade to roberta-large later if needed
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

# 2. Define the Preprocessing Function
def preprocess_mcq_data(examples):
    """
    Duplicates the prompt 5 times and pairs it with options A, B, C, D, E.
    """
    prompts = [[context] * 5 for context in examples["prompt"]]

    options = []
    for i in range(len(examples["prompt"])):
        # Extract the 5 options for each specific question
        opts = [
            examples["A"][i],
            examples["B"][i],
            examples["C"][i],
            examples["D"][i],
            examples["E"][i]
        ]
        options.append(opts)

    # Flatten the lists to feed into the tokenizer
    prompts_flat = sum(prompts, [])
    options_flat = sum(options, [])

    # Tokenize the pairs (Prompt + Option)
    tokenized_examples = tokenizer(
        prompts_flat,
        options_flat,
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    # Un-flatten back into shape: [batch_size, 5 (options), sequence_length]
    return {
        k: [v[i : i + 5] for i in range(0, len(v), 5)]
        for k, v in tokenized_examples.items()
    }

# 3. Prepare the Hugging Face Dataset
# Assuming 'train_df' is your loaded Pandas DataFrame
# Map string labels (A-E) to integer indices (0-4)
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train['label'] = train['answer'].map(label_map)

# Convert Pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(train)

# Apply the tokenization function
tokenized_dataset = hf_dataset.map(preprocess_mcq_data, batched=True)

In [ ]:
# 4. Define Evaluation Metrics (Standard Accuracy for now)
def compute_metrics(eval_predictions):
    predictions, label_ids = eval_predictions
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": (preds == label_ids).mean()}

# 5. Configure Training Arguments
training_args = TrainingArguments(
    output_dir="./roberta_mcq_model",
    eval_strategy="epoch",       # <-- CHANGED FROM evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none"
)

# 6. Initialize and Run the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    processing_class=tokenizer,      # <-- CHANGED FROM 'tokenizer=tokenizer'
    compute_metrics=compute_metrics,
)

print("Starting RoBERTa Fine-Tuning...")
trainer.train()

In [ ]:
# 1. Load the test data
hf_test_dataset = Dataset.from_pandas(test)

# 2. Tokenize using the same preprocessing function we built earlier
tokenized_test = hf_test_dataset.map(preprocess_mcq_data, batched=True)

# 3. Get raw predictions (logits) from the trainer
print("Generating predictions on test set...")
predictions = trainer.predict(tokenized_test).predictions

# 4. Extract the top 3 answers for each question
import numpy as np
label_map_reverse = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
final_predictions = []

for logits in predictions:
    # Get indices of the top 3 highest scores in descending order
    top_3_indices = np.argsort(logits)[-3:][::-1]
    
    # Map integer indices back to letters (e.g., [0, 2, 1] -> ['A', 'C', 'B'])
    top_3_letters = [label_map_reverse[idx] for idx in top_3_indices]
    
    # Join with spaces as required by the Kaggle format
    final_predictions.append(" ".join(top_3_letters))

In [ ]:
submission_df = pd.DataFrame({
    'id': test['id'],
    'Prediction': final_predictions
})

# Save to CSV
submission_df.to_csv("submission.csv", index=False)
print("Submission saved! Ready to upload to Kaggle.")